# Orchestrator

Примеры использования пайплайна:

In [1]:
import sys
import os
import json

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# Добавляем корень репозитория в путь, чтобы импортировать пакет app
sys.path.insert(0, os.path.abspath(".."))

from app.orchestrator.runner import run
from app.models.state import PipelineResult
from app.logger import setup_logging
from app.config import get_config
from app.generator.agent import GeneratorAgent

/Users/mishasdk/repo/mipt-secure-sql-agents/.venv/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
# Можно задать переменные окружения в .env, либо перезаписать тут.
os.environ["MODEL_NAME"] = "openai/gpt-4o-mini"

# Включаем логгирование.
setup_logging()

## Запуск пайплана целиком

In [4]:
query = "Show me all users who registered in the last 30 days"

result: PipelineResult = run(query)
result

2026-05-16 18:47:11 [INFO] app.generator: GeneratorAgent called | query='Show me all users who registered in the last 30 days'
2026-05-16 18:47:11 [INFO] app.generator.rag: tool=retriever | action=retrieve | query='Show me all users who registered in the last 30 days' | k=8
2026-05-16 18:47:11 [INFO] app.generator.rag: tool=embeddings | action=query | model=perplexity/pplx-embed-v1-4b
2026-05-16 18:47:12 [INFO] app.generator.rag: tool=embeddings | elapsed=0.96s
2026-05-16 18:47:12 [INFO] app.generator.rag: tool=retriever | result_tables=['participant_app', 'sys_company', 'scp_sec_check_res', 'scp_project_ans', 'mler_application', 'scp_application', 'sys_employee', 'dict_product']
2026-05-16 18:47:12 [INFO] app.generator: tool=retriever | elapsed=0.96s | tables=['participant_app', 'sys_company', 'scp_sec_check_res', 'scp_project_ans', 'mler_application', 'scp_application', 'sys_employee', 'dict_product']
2026-05-16 18:47:12 [INFO] app.generator: tool=llm | action=invoke | agent=generato

PipelineResult(final_sql='SELECT * FROM sys_employee;-- исправление синтаксиса, убрана ошибка, связанная с запятой после SELECT * (вставьте нужный код для заполнения запроса)  \n-- здесь должно быть продолжение запроса для обработки данных, чтобы избежать ошибок  \n-- добавьте нужные условия и операции (не указанные в предоставленном запросе) \n;  \n  \n  \n  \n  \n', approved=False, iterations_used=3, iteration_logs=[IterationLog(iteration=0, sql_query="SELECT id, name, create_date FROM sys_employee WHERE create_date >= NOW() - INTERVAL '30 days' LIMIT 1000", judge_output=JudgeOutput(verdict=<Verdict.REJECTED: 'REJECTED'>, risk_score=9.0, issues=[FoundIssue(type=<IssueType.POLICY: 'POLICY'>, severity=<Severity.HIGH: 'HIGH'>, message="Table 'sys_employee' is not in the allowed list", fix_instruction='Use only allowed tables: orders, products, users')]), timestamp=datetime.datetime(2026, 5, 16, 18, 47, 0, 432192)), IterationLog(iteration=1, sql_query="SELECT * FROM users WHERE status = 

In [ ]:
print(f"Approved: {result.approved}")
print(f"Iterations used: {result.iterations_used}")
print(f"\nFinal SQL:\n{result.final_sql}")

print("\nIteration logs:")
print(json.dumps([log.model_dump(mode="json") for log in result.iteration_logs], indent=2))

## Запуск Generator

In [ ]:
config = get_config()
llm = ChatOpenAI(
    base_url=config.openrouter_base_url,
    api_key=config.open_router_api_key,
    model=config.model_name,
)

generator = GeneratorAgent(llm)

state = {
    "messages": [HumanMessage(content="Show me all users who registered in the last 30 days")],
    "judge_responses": [],
    "llm_calls": 0,
    "audit_iteration": 0,
}

output = generator(state)
print(output)

Других агентов можно запускать по аналогии.